# Reconciliação
Umas queries de apoio pra tirar print pra pasta `docs/evidencias/`.

In [0]:
from pyspark.sql import functions as F

In [0]:
%sql
SELECT _ingestion_id, collection, stage, load_type, watermark_inicial,
       watermark_final, qtd_lida_origem, qtd_gravada_destino,
       start_time, end_time, duracao_seg, status, mensagem_erro
FROM meu_catalog.bronze.control_ingestion_log
ORDER BY start_time DESC
LIMIT 50

### Quanto leu x quanto gravou, por coleção

In [0]:
%sql
SELECT collection,
       SUM(qtd_lida_origem)  AS total_lido,
       SUM(qtd_gravada_destino) AS total_gravado,
       SUM(CASE WHEN status != 'SUCCESS' THEN 1 ELSE 0 END) AS execucoes_com_problema
FROM meu_catalog.bronze.control_ingestion_log
WHERE stage = 'bronze'
GROUP BY collection
ORDER BY collection

### Nulos em `_source_id` na Bronze

In [0]:
collections = ["movies", "comments", "users", "theaters", "sessions", "embedded_movies"]
for c in collections:
    table = f"meu_catalog.bronze.sample_mflix__{c}"
    if not spark.catalog.tableExists(table):
        print(f"{c}: tabela ainda não existe")
        continue
    df = spark.table(table)
    total = df.count()
    nulos = df.filter("_source_id IS NULL OR _source_id = ''").count()
    pct = round(100.0 * nulos / total, 4) if total else 0.0
    print(f"{c:<20} total={total:<8} nulos_chave={nulos:<6} pct={pct}%")

### Duplicou alguma coisa?

In [0]:
for c in collections:
    table = f"meu_catalog.bronze.sample_mflix__{c}"
    if not spark.catalog.tableExists(table):
        continue
    dup = (
        spark.table(table).groupBy("_source_id").count()
        .filter("count > 1").count()
    )
    print(f"{c:<20} _source_id duplicados = {dup}")

### O que caiu na quarentena

In [0]:
if spark.catalog.tableExists("meu_catalog.bronze.control_ingestion_quarantine"):
    display(
        spark.table("meu_catalog.bronze.control_ingestion_quarantine")
        .groupBy("motivo").count().orderBy(F.col("count").desc())
    )
else:
    print("Nenhum registro em quarentena até o momento.")